# Token Counting API: Estimación de Costes Antes de Enviar

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/apis-avanzadas/06-token-counting.ipynb)

Controla costes con la Token Counting API de Anthropic: cuenta tokens antes de enviar, estima el coste y gestiona presupuestos.

In [ ]:
# pip install anthropic
import anthropic, os
client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
print('Cliente listo')

## 1. Conteo básico de tokens

In [ ]:
resp = client.messages.count_tokens(
    model='claude-haiku-4-5-20251001',
    messages=[{'role': 'user', 'content': 'Explica el aprendizaje automático en 3 párrafos.'}]
)
print(f'Tokens de entrada: {resp.input_tokens}')

## 2. Conteo con system prompt y herramientas

In [ ]:
herramientas = [{
    'name': 'buscar_documentos',
    'description': 'Busca documentos en la base de conocimiento.',
    'input_schema': {
        'type': 'object',
        'properties': {'query': {'type': 'string'}},
        'required': ['query']
    }
}]

resp = client.messages.count_tokens(
    model='claude-haiku-4-5-20251001',
    system='Eres un asistente de investigación. Responde en español.',
    tools=herramientas,
    messages=[{'role': 'user', 'content': '¿Documentos sobre redes neuronales?'}]
)
print(f'Tokens con system + tools: {resp.input_tokens}')

## 3. TokenBudgetManager

In [ ]:
class TokenBudgetManager:
    PRECIOS = {
        'claude-haiku-4-5-20251001': (0.80, 4.00),
        'claude-sonnet-4-6':         (3.00, 15.00),
        'claude-opus-4-7':           (15.00, 75.00),
    }

    def __init__(self, limite: int = 100_000, modelo: str = 'claude-haiku-4-5-20251001'):
        self.limite = limite
        self.modelo = modelo
        self.usado = 0

    def puede_enviar(self, mensajes: list, system: str = '') -> tuple:
        kwargs = {'model': self.modelo, 'messages': mensajes}
        if system:
            kwargs['system'] = system
        tokens = client.messages.count_tokens(**kwargs).input_tokens
        p_in, p_out = self.PRECIOS.get(self.modelo, (0.80, 4.00))
        coste = (tokens * p_in + 500 * p_out) / 1_000_000
        return (self.usado + tokens) <= self.limite, tokens, coste

    def registrar(self, entrada: int, salida: int):
        self.usado += entrada + salida

mgr = TokenBudgetManager(limite=50_000)
msgs = [{'role': 'user', 'content': 'Resume el concepto de transformers.'}]
puede, tokens, coste = mgr.puede_enviar(msgs)
print(f'Tokens: {tokens} | Coste est.: ${coste:.6f} | Dentro del límite: {puede}')

## 4. Comparativa de coste por modelo

In [ ]:
EUR = 0.92
prompt = [{'role': 'user', 'content': 'Analiza ventajas y desventajas del fine-tuning vs RAG.'}]
precios = {'claude-haiku-4-5-20251001': (0.80, 4.00), 'claude-sonnet-4-6': (3.00, 15.00), 'claude-opus-4-7': (15.00, 75.00)}

print(f'{"Modelo":<35} {"Tokens":>8} {"USD":>12} {"EUR":>12}')
print('-' * 70)
for modelo, (p_in, p_out) in precios.items():
    t = client.messages.count_tokens(model=modelo, messages=prompt).input_tokens
    coste = (t * p_in + 500 * p_out) / 1_000_000
    print(f'{modelo:<35} {t:>8} ${coste:>11.6f} €{coste*EUR:>11.6f}')

## 5. Comparativa español vs inglés

In [ ]:
pares = [
    ('¿Qué es el aprendizaje automático?', 'What is machine learning?'),
    ('Explica las redes neuronales convolucionales', 'Explain convolutional neural networks'),
]
print(f'{"Español":<46} {"Inglés":<44} {"ES":>5} {"EN":>5} {"Ratio":>7}')
for es, en in pares:
    t_es = client.messages.count_tokens(model='claude-haiku-4-5-20251001', messages=[{'role': 'user', 'content': es}]).input_tokens
    t_en = client.messages.count_tokens(model='claude-haiku-4-5-20251001', messages=[{'role': 'user', 'content': en}]).input_tokens
    print(f'{es:<46} {en:<44} {t_es:>5} {t_en:>5} {t_es/t_en:>7.2f}x')

## Resumen

| Técnica | Cuándo usarla |
|---|---|
| `count_tokens()` básico | Evitar errores de límite de contexto |
| `TokenBudgetManager` | Controlar gasto en sesiones largas |
| Comparativa por modelo | Elegir modelo según coste/capacidad |
| Comparativa ES/EN | Decidir si traducir prompts para ahorrar |

**Siguiente**: [`05-prompt-caching.md`](../../apis-avanzadas/05-prompt-caching.md)